# RT-DETR-L (Ultralytics) — evaluation

Scores the checkpoint written by [`rtdetr_l_train.ipynb`](rtdetr_l_train.ipynb) (`models/rtdetr_l/weights/best.pt`) on the test split.

The best checkpoint (by Ultralytics' validation fitness) is run on the test images, resized to 640×640 (no augmentation), and scored with
the `EvaluationFramework`, exactly like the torchvision models. `model.predict` gets BGR arrays at 640×640 with
`conf=0.001`; latency is the sum of Ultralytics' pre-process, inference and post-process times.

In [1]:
import numpy as np
from ultralytics import RTDETR

from ppe_detection_evaluation import RTDETR_L, EvaluationFramework
from ppe_detection_evaluation.classes import DEFAULT_CLASS_NAMES
from ppe_detection_evaluation.dataset.core import check_dataset, list_images, load_eval_example, xyxy_to_cxcywh
from ppe_detection_evaluation.experiment import DATA_ROOT, IMAGE_SIZE, MODELS_DIR, RESULTS_DIR

check_dataset(DATA_ROOT)  # the training notebook downloads the dataset

{'train': 1132, 'valid': 143, 'test': 141}

In [2]:
checkpoint = MODELS_DIR / "rtdetr_l" / "weights" / "best.pt"
model = RTDETR(checkpoint)
if tuple(model.names.values()) != DEFAULT_CLASS_NAMES:  # e.g. a checkpoint trained on another class set
    raise ValueError(f"{checkpoint} predicts {tuple(model.names.values())}, expected {DEFAULT_CLASS_NAMES}")

In [3]:
fw = EvaluationFramework(RTDETR_L)
for path in list_images(DATA_ROOT, "test"):
    image, gt_boxes, gt_labels = load_eval_example(path)
    result = model.predict(
        np.ascontiguousarray(image[..., ::-1]),  # predict() expects BGR arrays
        imgsz=IMAGE_SIZE,
        conf=fw.score_threshold,                 # 0.001, the same detection threshold as the other models
        verbose=False,
    )[0]
    fw.add(
        path.stem,
        result.boxes.xyxy.cpu().numpy(),
        result.boxes.conf.cpu().numpy(),
        result.boxes.cls.cpu().numpy(),
        xyxy_to_cxcywh(gt_boxes, IMAGE_SIZE, IMAGE_SIZE),
        gt_labels,
        latency_ms=sum(result.speed.values()),   # Ultralytics' pre-process + inference + post-process time
    )

results = fw.evaluate()
fw.save(RESULTS_DIR / "rtdetr_l_test.json")

PosixPath('/home/llravelo/Documents/projects/42177-image-processing-ppe-detection/results/rtdetr_l_test.json')

In [4]:
det = results["detection"]["groups"]
print(f"mAP@[.5:.95] {det['all']['mAP']:.4f}   mAP@.5 {det['all']['mAP50']:.4f}")
print(f"PPE mAP@.5 {det['ppe']['mAP50']:.4f}   small PPE mAP@.5 {results['detection']['by_size']['small']['groups']['ppe']['mAP50']:.4f}")
print("violation mean:", results["violation"]["mean"])
print("latency:", results["latency"])

mAP@[.5:.95] 0.3868   mAP@.5 0.6809
PPE mAP@.5 0.6386   small PPE mAP@.5 0.2398
violation mean: {'precision_at_recall': 0.0, 'recall_at_fppi': 0.3096153846153846, 'lamr': 0.6420172514630023, 'max_recall': 0.7317307692307693, 'max_f1': 0.46565656565656566}
latency: {'num_images': 141, 'mean_ms': 15.68702967385776, 'std_ms': 1.7279540336909323, 'p50_ms': 15.4372399993008, 'p95_ms': 16.867848000401864, 'fps': 63.74693111382887}
